# Check partial-label loss gradients

Compare `diceCELoss(domain_adaptation=True)` with `diceCELoss_gradient_fix` on small synthetic examples. The old loss receives **softmax probabilities**; the new loss receives **raw logits** and masks unavailable classes before softmax. `-1` means unavailable, `0` means an available negative class, and `1` identifies the known target class.

Run all cells with the repository's TensorFlow environment. Plots use native VS Code Plotly rendering, without widgets. This checks the objective and gradients; it does not train a model or assess segmentation performance.

In [1]:
from pathlib import Path
from collections.abc import Callable
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

current_directory: Path = Path.cwd().resolve()
repository_root: Path = current_directory
for candidate in (current_directory, *current_directory.parents):
    if (candidate / "src/evaluation/losses.py").is_file():
        repository_root = candidate
        break
else:
    raise FileNotFoundError("Run this notebook inside the RNET3D repository")

sys.path.insert(0, str(repository_root))
from src.evaluation.losses import diceCELoss, diceCELoss_gradient_fix, masked_softmax_gradient_fix
from src.models.networks import mt_r_net_3d

renderer: str = "vscode"

## One partially annotated voxel

Use the 11-channel ordering from the domain-adaptation example. Only channels 1, 3, 7 and 10 are available. Channel 1 is the target. The zero initial logits make the denominator effect easy to see.

In [2]:
label_ids: list[int] = [0, 1, 2, 3, 5, 6, 12, 13, 15, 16, 21]
target: tf.Tensor = tf.constant([[[[[-1., 1., -1., 0., -1., -1., -1., 0., -1., -1., 0.]]]]])
initial_logits: np.ndarray = np.zeros(target.shape, dtype=np.float32)
available: np.ndarray = target.numpy().reshape(-1) != -1

gamma: float = 0.8
smooth: float = 1e-5
legacy_loss: Callable = diceCELoss(gamma=gamma, smooth=smooth, batch_wise=True, domain_adaptation=True)
fixed_loss: Callable = diceCELoss_gradient_fix(gamma=gamma, smooth=smooth, batch_wise=True)


def evaluate(values: np.ndarray, labels: tf.Tensor, fixed: bool) -> tuple[float, np.ndarray, np.ndarray]:
    """Return loss, probabilities and derivatives with respect to RAW logits."""
    logits: tf.Variable = tf.Variable(values)

    with tf.GradientTape() as tape:
        if fixed:
            value: tf.Tensor = fixed_loss(labels, logits)
        else:
            probabilities: tf.Tensor = tf.nn.softmax(logits, axis=-1)
            value = legacy_loss(labels, probabilities)

    gradient: tf.Tensor = tape.gradient(value, logits)
    if fixed:
        probabilities = masked_softmax_gradient_fix(labels, logits)
    else:
        probabilities = tf.nn.softmax(logits, axis=-1)

    return float(value.numpy()), probabilities.numpy(), gradient.numpy()


old_value, old_probability, old_gradient = evaluate(initial_logits, target, fixed=False)
new_value, new_probability, new_gradient = evaluate(initial_logits, target, fixed=True)
comparison: pd.DataFrame = pd.DataFrame({
    "channel": np.arange(len(label_ids)),
    "original label ID": label_ids,
    "available": available,
    "old probability": old_probability.reshape(-1),
    "conditional probability": new_probability.reshape(-1),
    "old dL/dlogit": old_gradient.reshape(-1),
    "fixed dL/dlogit": new_gradient.reshape(-1),
})
display(comparison)
print(f"Old loss: {old_value:.8f}; fixed loss: {new_value:.8f}")

np.testing.assert_array_equal(new_gradient.reshape(-1)[~available], 0)
assert np.any(np.abs(old_gradient.reshape(-1)[~available]) > 1e-5)
assert np.any(np.abs(new_gradient.reshape(-1)[available]) > 1e-5)

figure: go.Figure = go.Figure()
figure.add_bar(x=comparison["channel"], y=comparison["old dL/dlogit"], name="Original")
figure.add_bar(x=comparison["channel"], y=comparison["fixed dL/dlogit"], name="Gradient fix")
figure.update_layout(title="Logit gradients: only channels 1, 3, 7, 10 are available", xaxis_title="Channel", yaxis_title="dL/dlogit", barmode="group")
figure.show(renderer=renderer)

,channel,original label ID,available,old probability,conditional probability,old dL/dlogit,fixed dL/dlogit
0,0,0,False,0.090909,0.00,0.006863,0.000000
1,1,1,True,0.090909,0.25,-0.068642,-0.162003
2,2,2,False,0.090909,0.00,0.006863,0.000000
3,3,3,True,0.090909,0.25,0.006867,0.054001
4,4,5,False,0.090909,0.00,0.006863,0.000000
5,5,6,False,0.090909,0.00,0.006863,0.000000
6,6,12,False,0.090909,0.00,0.006863,0.000000
7,7,13,True,0.090909,0.25,0.006867,0.054001
8,8,15,False,0.090909,0.00,0.006863,0.000000
9,9,16,False,0.090909,0.00,0.006863,0.000000


Old loss: 0.11680433; fixed loss: 0.45724624


## Change an unavailable logit

Changing channel 2 changes the original valid-class probability and loss through the shared softmax denominator. Both remain constant with the fix. Absolute loss values are not a quality comparison: the corrected reduction also excludes unavailable entries.

In [3]:
rows: list[dict] = []
for ignored_logit in np.linspace(-8, 8, 33):
    changed: np.ndarray = initial_logits.copy()
    changed[..., 2] = ignored_logit
    old_loss_value, old_prob, _ = evaluate(changed, target, fixed=False)
    fixed_loss_value, fixed_prob, _ = evaluate(changed, target, fixed=True)
    rows.append({"ignored logit": ignored_logit, "old loss": old_loss_value, "fixed loss": fixed_loss_value,
                 "old target probability": old_prob[..., 1].item(), "fixed target probability": fixed_prob[..., 1].item()})

sweep: pd.DataFrame = pd.DataFrame(rows)
np.testing.assert_array_equal(sweep["fixed loss"].to_numpy(), new_value)
np.testing.assert_array_equal(sweep["fixed target probability"].to_numpy(), 0.25)

figure = make_subplots(rows=1, cols=2, subplot_titles=("Loss", "Probability of known target"))
for prefix in ("old", "fixed"):
    loss_trace = go.Scatter(x=sweep["ignored logit"], y=sweep[f"{prefix} loss"], name=prefix)
    probability_trace = go.Scatter(x=sweep["ignored logit"], y=sweep[f"{prefix} target probability"], name=prefix, showlegend=False)
    figure.add_trace(loss_trace, row=1, col=1)
    figure.add_trace(probability_trace, row=1, col=2)
figure.update_xaxes(title_text="Unavailable logit (channel 2)")
figure.show(renderer=renderer)

for extreme in (-10000., 10000.):
    changed = initial_logits.copy()
    changed[..., 2] = extreme
    value, _, gradient = evaluate(changed, target, fixed=True)
    assert value == new_value
    np.testing.assert_array_equal(gradient.reshape(-1)[~available], 0)
print("Unavailable logits have no effect, including extreme values.")

Unavailable logits have no effect, including extreme values.


## Independent finite-difference check

Compare automatic differentiation with a central finite difference for one available and one unavailable logit.

In [4]:
epsilon: float = 1e-3
checks: list[dict] = []
for fixed in (False, True):
    _, _, analytical = evaluate(initial_logits, target, fixed=fixed)
    for channel in (1, 2):
        plus = initial_logits.copy()
        minus = initial_logits.copy()
        plus[..., channel] += epsilon
        minus[..., channel] -= epsilon
        upper, _, _ = evaluate(plus, target, fixed=fixed)
        lower, _, _ = evaluate(minus, target, fixed=fixed)
        numerical: float = (upper - lower) / (2 * epsilon)
        derivative: float = analytical[..., channel].item()
        np.testing.assert_allclose(numerical, derivative, atol=5e-5, rtol=0.01)
        checks.append({"fixed": fixed, "channel": channel, "autodiff": derivative, "finite difference": numerical})
display(pd.DataFrame(checks))

,fixed,channel,autodiff,finite difference
0,False,1,-0.068642,-0.068624
1,False,2,0.006863,0.006869
2,True,1,-0.162003,-0.162005
3,True,2,0.000000,0.000000


## Fully annotated, mixed, and unknown targets

Fully annotated inputs retain the original computation. An unavailable channel can still be supervised in another sample. All-ignored voxels, and voxels with no known positive target, contribute zero loss and gradient. The latter policy avoids forcing an unknown true class into a known class, but supplies **no negative-only supervision** at these voxels.

In [5]:
rng: np.random.Generator = np.random.default_rng(7)
class_indices: np.ndarray = rng.integers(0, 11, size=(2, 2, 2, 2))
full_target: tf.Tensor = tf.one_hot(class_indices, depth=11)
full_logits: np.ndarray = rng.normal(size=full_target.shape).astype(np.float32)
full_old, _, gradient_old = evaluate(full_logits, full_target, fixed=False)
full_new, _, gradient_new = evaluate(full_logits, full_target, fixed=True)
np.testing.assert_allclose(full_new, full_old, atol=1e-7, rtol=1e-6)
np.testing.assert_allclose(gradient_new, gradient_old, atol=1e-7, rtol=1e-6)

for unknown_values in ([-1., -1., -1.], [0., 0., -1.]):
    unknown_target = tf.constant([[[[unknown_values]]]])
    unknown_logits = np.zeros(unknown_target.shape, dtype=np.float32)
    value, _, gradient = evaluate(unknown_logits, unknown_target, fixed=True)
    assert value == 0
    np.testing.assert_array_equal(gradient, 0)

mixed_target = tf.constant([[[[[1., 0., -1.]]]], [[[[0., 0., 1.]]]]])
mixed_logits = np.zeros(mixed_target.shape, dtype=np.float32)
_, _, mixed_gradient = evaluate(mixed_logits, mixed_target, fixed=True)
assert mixed_gradient[0, 0, 0, 0, 2] == 0
assert abs(mixed_gradient[1, 0, 0, 0, 2]) > 1e-5

graph_loss = tf.function(fixed_loss)
graph_logits = tf.Variable(initial_logits)
with tf.GradientTape() as tape:
    graph_value = graph_loss(target, graph_logits)
graph_gradient = tape.gradient(graph_value, graph_logits).numpy()
np.testing.assert_allclose(graph_value.numpy(), new_value, atol=1e-7)
np.testing.assert_array_equal(graph_gradient.reshape(-1)[~available], 0)
print("PASS: fully annotated equivalence, unknown voxels, mixed annotations, and graph-mode gradients.")

PASS: fully annotated equivalence, unknown voxels, mixed annotations, and graph-mode gradients.


## Use logits from the existing network

The existing builder and saved inference models still produce `regions` softmax probabilities. **Do not attach the new loss directly to that output.** This small example creates a training view from the tensor immediately before `regions`, sharing the same weights. The original model remains the probability-producing inference view.

The linear output keeps the `regions` name and float32 dtype. Other heads are unchanged. Existing ROI metrics expect probabilities; do not attach them unchanged to the logits output. Auxiliary heads can still update shared weights: the zero-gradient claim concerns the ROI loss derivative with respect to unavailable ROI logits.

In [6]:
spatial_shape: tuple[int, int, int] = (16, 16, 16)
filter_widths: list[int] = [1, 2, 4, 8, 16]
inference_model: tf.keras.Model = mt_r_net_3d(spatial_shape, 1, 11, filter_widths, with_classifier=True)
regions_layer = inference_model.get_layer("regions")
assert regions_layer.get_config()["activation"] == "softmax"

linear_regions = tf.keras.layers.Activation("linear", dtype="float32", name="regions")
roi_logits = linear_regions(regions_layer.input)
training_outputs = [roi_logits, inference_model.get_layer("brain_mask").output, inference_model.get_layer("classifier").output]
training_model = tf.keras.Model(inference_model.inputs, training_outputs, name="rnet_gradient_fix")

inputs = tf.zeros((1, 16, 16, 16, 1), dtype=tf.float32)
inference_outputs = inference_model(inputs, training=False)
logit_outputs = training_model(inputs, training=False)
restored_probabilities = tf.nn.softmax(logit_outputs[0], axis=-1)
np.testing.assert_allclose(restored_probabilities.numpy(), inference_outputs[0].numpy(), atol=1e-7)
assert training_model.output_names == inference_model.output_names
assert {id(weight) for weight in training_model.weights} == {id(weight) for weight in inference_model.weights}

volume_target = tf.tile(target, [1, 16, 16, 16, 1])
with tf.GradientTape() as tape:
    outputs = training_model(inputs, training=False)
    roi_value = fixed_loss(volume_target, outputs[0])
roi_gradient = tape.gradient(roi_value, outputs[0]).numpy()
np.testing.assert_array_equal(roi_gradient[..., ~available], 0)
assert np.any(np.abs(roi_gradient[..., available]) > 0)
print("PASS: unchanged inference probabilities, shared weights, and zero ignored-logit gradients in the real network.")
print("Training outputs:", training_model.output_names)

PASS: unchanged inference probabilities, shared weights, and zero ignored-logit gradients in the real network.
Training outputs: ['regions', 'brain_mask', 'classifier']


The training view can be compiled with `diceCELoss_gradient_fix(...)` for `regions` and the existing losses for the other heads. Saving this training view saves a **logits-output** model; keep using/saving the original inference view when probabilities are required. The views share trained weights in this process.

The corrected objective uses conditional probabilities over available classes and excludes unsupervised entries from reductions. It preserves the historical Dice smoothing and CE mean-over-classes convention; consequently it does not address every other loss/metric behavior documented in `docs/losses_metrics.md`. A sample with only one available class provides no class-competition signal. Gradient checks establish the masking contract, not improved segmentation: assessing the corrected training protocol requires later retraining and evaluation.